# Download and prepare `SW_IN` (2004-2025)

**notebook version**: `4` (16 Jul 2026)
**new in this version**:
- database access switched from the standalone `dbc-influxdb` package to diive's in-house
  InfluxDB engine [`InfluxIO`](../../../../diive/diive/core/io/db/influx/influxio.py)
  (`dbc-influxdb` is now part of diive)
- the 30MIN time resolution check is now done by `download(..., verify_freq=...)`
- gap-filling switched to diive's purpose-built `SWINGapFillerXGBoost`, which handles the
  daytime/nighttime split, the nighttime zero-offset correction and the physical
  floor of zero internally (this replaces the manual `XGBoostTS` +
  `remove_radiation_zero_offset` + "set negatives to zero" steps of version `3`)
- short daytime gaps (<= 1 h) are now filled by clearness-index interpolation
  (`interpolate_short_gaps=2`) instead of by the model, which is blind to the target's
  own neighbours
- import paths updated to the reorganized diive package layout (`diive.pkgs.*` is gone)

## ℹ️ About this notebook

Downloads screened `SW_IN` from the InfluxDB database, merges the three data sources into one
continuous series, corrects a known logger timestamp shift, gap-fills the result and stores it
to the external data folder for use by notebook `11`.

Downloading uses diive's in-house InfluxDB engine `InfluxIO`, which needs `influxdb-client`
to be installed in the environment.

***

# Info about data sources

- `SW_IN`: NABEL (2004-2018), mst (2005-2021), diive (2022-2025)
- ETH data are the main data
- NABEL data are used to gap-fill ETH data to create a complete time series 2004-2018
- ETH data 2019-2025 is gap-filled separately, using only `SW_IN_POT`+
- Gap-filling is done using XGBoost, it also uses e.g. timestamp and potential radiation as features

Legend:
- NABEL ... Data from [NABEL](https://www.bafu.admin.ch/bafu/en/home/topics/air/luftbelastung/national-air-pollution-monitoring-network--nabel-.html), meteoscreening with [diive](https://github.com/holukas/diive)
- mst ... Data from ETH, meteoscreening with the now deprecated MeteoscreeningTool
- diive ... Data from ETH, meteoscreening with diive

## ⏱️ Timestamp convention

The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`**. On download, `InfluxIO`
shifts the UTC timestamp by `TIMEZONE_OFFSET_TO_UTC_HOURS`, so the returned data are in **local
time** (UTC + offset) and still `TIMESTAMP_END`. `TimestampSanitizer(output_middle_timestamp=True)`
then converts to `TIMESTAMP_MID`, which is what the rest of this notebook works with.

`START` and `STOP` are interpreted in the same timezone as `TIMEZONE_OFFSET_TO_UTC_HOURS`.

***

# ⚙️ Settings

## Data settings

In [ ]:
DIRCONF = r'F:\Sync\luhk_work\20 - CODING\22 - POET\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'  # Folder with configuration files: needed e.g. for connection to database
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # Timezone, e.g. "1" is translated to timezone "UTC+01:00" (CET, winter time)
REQUIRED_TIME_RESOLUTION = '30min'  # 30MIN time resolution
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389  # CH-LAE

# Output: data files live in the external (untracked) data folder, with the same
# numeric prefix as this notebook.
OUTNAME = "01_METEO_SW_IN_GAPFILLED_2004-2025"
OUTPATH = r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO"

## Imports

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib.metadata

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme('notebook')

import diive as dv
from diive.core.io.db.influx import InfluxIO  # diive's in-house InfluxDB engine (needs influxdb-client)
from diive.core.times.times import TimestampSanitizer

import warnings

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Last run: {dt_string}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Small helper: `HeatmapDateTime` no longer takes `title`/`ax` in its constructor, so this wraps
the two calls that every heatmap below needs.

In [ ]:
def heatmap(series, ax, title, zlabel=r'$\mathrm{W\ m^{-2}}$'):
    """Plot *series* as a date/time heatmap onto *ax* and set its title."""
    dv.plotting.HeatmapDateTime(series=series).plot(ax=ax, cb_digits_after_comma=0, zlabel=zlabel)
    ax.set_title(title, fontsize=11, fontweight='bold')


# Result flags of SWINGapFillerXGBoost, see the class docstring.
FLAG_LEGEND = {
    0: '0 = observed',
    1: '1 = daytime gap, filled by the XGBoost model',
    2: '2 = daytime gap, filled by the timestamp-only fallback model (a driver was missing)',
    3: '3 = nighttime gap, set to zero by physics',
    4: '4 = daytime gap, filled by clearness-index interpolation',
}

## 🔌 Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

***

# ⬇️ `NABEL` data (2004-2018) and ETH data (2022-2025) from `diive` meteoscreening

## Download

In [ ]:
%%time

BUCKET = f'ch-lae_processed'
FIELDS = ['SW_IN_NABEL_T1_49_1', 'SW_IN_T1_47_1']
MEASUREMENTS = ['SW']
START = '2004-01-01 00:00:01'
STOP = '2026-01-01 00:00:01'
DATA_VERSION = 'meteoscreening_diive'

nabel_eth_diive_swin_2004_2018_2022_2025, _, _ = dbc.download(
    bucket=BUCKET,
    measurements=MEASUREMENTS,
    fields=FIELDS,
    start=START,  # Download data starting with this date (the start date itself IS included),
    stop=STOP,  # Download data before this date (the stop date itself IS NOT included),
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
    verify_freq=REQUIRED_TIME_RESOLUTION  # warns if the downloaded data are not 30MIN
)

In [ ]:
nabel_eth_diive_swin_2004_2018_2022_2025

## Sanitize timestamp

In [ ]:
nabel_eth_diive_swin_2004_2018_2022_2025 = TimestampSanitizer(data=nabel_eth_diive_swin_2004_2018_2022_2025, output_middle_timestamp=True).get()
nabel_eth_diive_swin_2004_2018_2022_2025

***

# ⬇️ Data from `mst` meteoscreening (2004-2021)

## Download

In [ ]:
%%time

BUCKET = f'ch-lae_processed'
FIELDS = ['SW_IN_T1_47_1']
MEASUREMENTS = ['SW']
START = '2004-01-01 00:00:01'
STOP = '2022-01-01 00:00:01'
DATA_VERSION = 'meteoscreening_mst'

mst_swin_2004_2021, _, _ = dbc.download(
    bucket=BUCKET,
    measurements=MEASUREMENTS,
    fields=FIELDS,
    start=START,  # Download data starting with this date (the start date itself IS included),
    stop=STOP,  # Download data before this date (the stop date itself IS NOT included),
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
    verify_freq=REQUIRED_TIME_RESOLUTION  # warns if the downloaded data are not 30MIN
)

In [ ]:
mst_swin_2004_2021

## Sanitize timestamp

In [ ]:
mst_swin_2004_2021 = TimestampSanitizer(data=mst_swin_2004_2021, output_middle_timestamp=True).get()
mst_swin_2004_2021

***

# ❇️ Merge data

ETH data from `diive` meteoscreening (2022-2025) are the main data, gaps before 2022 are
filled in from the `mst` meteoscreening.

In [ ]:
# Merge data on index
swin_2004_2025 = nabel_eth_diive_swin_2004_2018_2022_2025.copy()
swin_2004_2025['SW_IN_T1_47_1'] = swin_2004_2025['SW_IN_T1_47_1'].combine_first(mst_swin_2004_2021['SW_IN_T1_47_1'])
swin_2004_2025

## Sanitize timestamp

In [ ]:
swin_2004_2025 = TimestampSanitizer(data=swin_2004_2025, output_middle_timestamp=True).get()
swin_2004_2025

## Heatmap

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(14, 10), dpi=100, layout="constrained")
fig.suptitle(f'Half-hourly', fontsize=16)
heatmap(swin_2004_2025['SW_IN_T1_47_1'], ax=axs[0], title="SW_IN_T1_47_1")
heatmap(swin_2004_2025['SW_IN_NABEL_T1_49_1'], ax=axs[1], title="SW_IN_NABEL_T1_49_1")

***

# ✴️ Corrections

### Timestamp shift in August 2012
**Info from fieldbook entry 17 Aug 2012:**
>   adjusted the logger date and time. On Servertime (Computer System Time) 17.08.2012 12:31.42 the loggertime was 16.08.2012 08:06:00. Synchronized time at 17.08.2012 11:40:00 servertime. The logger data aquisition on the moxa embedded was restarted

In [ ]:
AFFECTED_VARS = ['SW_IN_T1_47_1']
_df = swin_2004_2025.copy()

for av in AFFECTED_VARS:
    fig = plt.figure(figsize=(24, 6), dpi=72)
    fig.suptitle(f"{av}")
    gs = gridspec.GridSpec(1, 5)  # rows, cols
    gs.update(wspace=0.5, hspace=1, left=.1, right=.9, top=.85, bottom=.1)
    ax_before = fig.add_subplot(gs[0, 0])
    ax_unshifted = fig.add_subplot(gs[0, 1])
    ax_shifted = fig.add_subplot(gs[0, 2])
    ax_after = fig.add_subplot(gs[0, 3])
    ax_corrected = fig.add_subplot(gs[0, 4])

    # Show time period around issue, before correction
    _show_locs = (_df.index >= '2012-08-01 00:00') & (_df.index <= '2012-09-01 00:00')
    heatmap(_df.loc[_show_locs, av], ax=ax_before, title="Before correction")

    # Get shifted time period
    _series_corrected = _df.loc[_show_locs, av].copy()
    # Identify shifted time period
    ISSUE_START = '2012-08-09 00:00'
    ISSUE_END = '2012-08-17 00:00'
    _shifted_locs = (_df.index >= ISSUE_START) & (_df.index <= ISSUE_END)
    _series_shifted = _df.loc[_shifted_locs, av].copy()
    _series_shifted = _series_shifted.dropna()
    heatmap(_series_shifted, ax=ax_unshifted, title="UNSHIFTED time period")

    # Shift SW_IN by 15.5 hours during shifted time period, create corrected time series
    _series_shifted.index = _series_shifted.index + pd.Timedelta(hours=15.5)
    heatmap(_series_shifted, ax=ax_shifted, title="SHIFTED time period")

    # Delete data between start of issue and the last timestamp of shifted data
    _overwrite_locs = (_df.index >= ISSUE_START) & (_df.index <= _series_shifted.index[-1])
    _df.loc[_overwrite_locs, av] = np.nan
    heatmap(_df.loc[_show_locs, av], ax=ax_after, title="After deletion")

    # Fill in corrected values
    _df.loc[_overwrite_locs, av] = _series_shifted
    heatmap(_df.loc[_show_locs, av], ax=ax_corrected, title="After correction")

In [ ]:
swin_2004_2025 = _df.copy()
swin_2004_2025

::: {.callout-note title="Zero-offset correction"}
Version `3` of this notebook removed the `SW_IN` nighttime zero-offset here, with an explicit call
to `remove_radiation_zero_offset()`. That step is now done inside the gap-filler via
`correct_nighttime_offset=True` (see below), which applies the same correction
(`remove_nighttime_zero_offset()`) to each series before modelling. It is therefore no longer
done separately at this point.
:::

## Dataframe

In [ ]:
display(swin_2004_2025)
swin_2004_2025.describe()

***

# 🟪 Prepare dataframes

Split into the two periods that are gap-filled separately: 2004-2018 (NABEL available as
driver) and 2019-2025 (no NABEL).

In [ ]:
swin_2004_2018 = swin_2004_2025.loc[swin_2004_2025.index.year <= 2018].copy()
swin_2004_2018

In [ ]:
swin_2019_2025 = swin_2004_2025.loc[swin_2004_2025.index.year >= 2019].copy()
swin_2019_2025 = swin_2019_2025.drop('SW_IN_NABEL_T1_49_1', axis=1)
swin_2019_2025

***

# 🟥 Gap-filling NABEL (2004-2018)

`SWINGapFillerXGBoost` calculates `SW_IN_POT` from `SITE_LAT`/`SITE_LON` itself, uses it to split
daytime from nighttime, sets nighttime gaps to zero and fills daytime gaps with XGBoost trained on
daytime observations only. `correct_nighttime_offset=True` removes the sensor's nighttime
zero-offset first, and negative values are clipped to zero internally
(`below_zero='zero'`) — no manual post-correction needed.

`interpolate_short_gaps=2` fills daytime gaps of up to 2 records (1 h at 30MIN resolution) by
interpolating the clearness index (`SW_IN`/`SW_IN_POT`) instead of using the model, and never
bridges a night. Short gaps are exactly the case the model cannot see, because the feature
engineer excludes the target from its own features.

## Run gap-filling

In [ ]:
%%time

TARGET = "SW_IN_NABEL_T1_49_1"

gf_nabel = dv.gapfilling.SWINGapFillerXGBoost(
    series=swin_2004_2018[TARGET],
    lat=SITE_LAT,
    lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
    correct_nighttime_offset=True,  # replaces the manual remove_radiation_zero_offset() step
    interpolate_short_gaps=2,  # fill daytime gaps <= 1h by clearness-index interpolation
    reduce_features=True,
    verbose=1,
    # XGBoost hyperparameters
    n_estimators=3000,
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
)
gf_nabel.run()

## Report

In [ ]:
gf_nabel.report()

## Results: scores and feature importances

In [ ]:
r = gf_nabel.results
print("Daytime model, held-out (train/test) scores:")
display(pd.Series(r.scores_traintest))
print("Flag counts:")
display(r.flag.value_counts().sort_index().rename(index=FLAG_LEGEND))

In [ ]:
r.feature_importances

## Add gap-filled series to dataframe

The gap-filler returns the series under its original name, so it is renamed to the `_gfXG`
convention used by the downstream notebooks.

In [ ]:
gapfilled_nabel = gf_nabel.results.gapfilled.rename(f"{TARGET}_gfXG")
swin_2004_2018[gapfilled_nabel.name] = gapfilled_nabel
swin_2004_2018[[TARGET, gapfilled_nabel.name]].describe()

## Plot time series

In [ ]:
swin_2004_2018[[TARGET, gapfilled_nabel.name]].plot(subplots=True, x_compat=True);

***

# 🟥 Gap-filling ETH (2004-2018) using NABEL+
- NABEL is now complete (no gaps) 2004-2018 and can be used as feature in those years

## Run gap-filling

The gap-filled NABEL series is passed in as an additional driver via `context_df`; `SW_IN_POT`
and the timestamp features are added by the gap-filler itself.

In [ ]:
%%time

TARGET = "SW_IN_T1_47_1"

gf_eth_2004_2018 = dv.gapfilling.SWINGapFillerXGBoost(
    series=swin_2004_2018[TARGET],
    lat=SITE_LAT,
    lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
    context_df=swin_2004_2018[['SW_IN_NABEL_T1_49_1_gfXG']],  # NABEL as additional driver
    correct_nighttime_offset=True,
    interpolate_short_gaps=2,  # fill daytime gaps <= 1h by clearness-index interpolation
    reduce_features=True,
    verbose=1,
    # XGBoost hyperparameters
    n_estimators=3000,
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
)
gf_eth_2004_2018.run()

## Report

In [ ]:
gf_eth_2004_2018.report()

## Results: scores and feature importances

In [ ]:
r = gf_eth_2004_2018.results
print("Daytime model, held-out (train/test) scores:")
display(pd.Series(r.scores_traintest))
print("Flag counts:")
display(r.flag.value_counts().sort_index().rename(index=FLAG_LEGEND))

In [ ]:
r.feature_importances

## Add gap-filled series to dataframe

In [ ]:
gapfilled_eth_2004_2018 = gf_eth_2004_2018.results.gapfilled.rename(f"{TARGET}_gfXG")
swin_2004_2018[gapfilled_eth_2004_2018.name] = gapfilled_eth_2004_2018
swin_2004_2018[[TARGET, gapfilled_eth_2004_2018.name]].describe()

## Dataframe

In [ ]:
display(swin_2004_2018)
swin_2004_2018.describe()

***

# 🟥 Plots (2004-2018)

## Heatmap

In [ ]:
fig, axs = plt.subplots(ncols=4, figsize=(16, 8), dpi=100, layout="constrained")
fig.suptitle(f'Half-hourly', fontsize=16)
heatmap(swin_2004_2018['SW_IN_T1_47_1'], ax=axs[0], title="SW_IN_T1_47_1")
heatmap(swin_2004_2018['SW_IN_T1_47_1_gfXG'], ax=axs[1], title="SW_IN_T1_47_1_gfXG")
heatmap(swin_2004_2018['SW_IN_NABEL_T1_49_1'], ax=axs[2], title="SW_IN_NABEL_T1_49_1")
heatmap(swin_2004_2018['SW_IN_NABEL_T1_49_1_gfXG'], ax=axs[3], title="SW_IN_NABEL_T1_49_1_gfXG")

## Time series plot

In [ ]:
swin_2004_2018.plot(subplots=True, x_compat=True);

***

# 🟧 Gap-filling ETH (2019-2025)
- From mst (2019-2021) and diive meteoscreening (2022-2025)
- No NABEL data in this period, so only `SW_IN_POT`+ timestamp features are used

## Run gap-filling

In [ ]:
%%time

TARGET = "SW_IN_T1_47_1"

gf_eth_2019_2025 = dv.gapfilling.SWINGapFillerXGBoost(
    series=swin_2019_2025[TARGET],
    lat=SITE_LAT,
    lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
    correct_nighttime_offset=True,
    interpolate_short_gaps=2,  # fill daytime gaps <= 1h by clearness-index interpolation
    reduce_features=True,
    verbose=1,
    # XGBoost hyperparameters
    n_estimators=3000,
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
)
gf_eth_2019_2025.run()

## Report

In [ ]:
gf_eth_2019_2025.report()

## Results: scores and feature importances

In [ ]:
r = gf_eth_2019_2025.results
print("Daytime model, held-out (train/test) scores:")
display(pd.Series(r.scores_traintest))
print("Flag counts:")
display(r.flag.value_counts().sort_index().rename(index=FLAG_LEGEND))

In [ ]:
r.feature_importances

## Add gap-filled series to dataframe

In [ ]:
gapfilled_eth_2019_2025 = gf_eth_2019_2025.results.gapfilled.rename(f"{TARGET}_gfXG")
swin_2019_2025[gapfilled_eth_2019_2025.name] = gapfilled_eth_2019_2025
swin_2019_2025[[TARGET, gapfilled_eth_2019_2025.name]].describe()

## Plot time series

In [ ]:
swin_2019_2025[[TARGET, gapfilled_eth_2019_2025.name]].plot(subplots=True, x_compat=True);

## Dataframe

In [ ]:
display(swin_2019_2025)
swin_2019_2025.describe()

***

# ➡️ Prepare dataframe for export

In [ ]:
export_df = pd.concat([swin_2004_2018, swin_2019_2025], axis=0)
export_df = export_df[['SW_IN_T1_47_1_gfXG']].copy()
export_df

Sanity check: the exported series must be complete (no gaps) and non-negative.

In [ ]:
print(f"Records:        {len(export_df)}")
print(f"Missing values: {export_df['SW_IN_T1_47_1_gfXG'].isnull().sum()}")
print(f"Negative values: {(export_df['SW_IN_T1_47_1_gfXG'] < 0).sum()}")
export_df.describe()

In [ ]:
fig, axs = plt.subplots(ncols=1, figsize=(4, 8), dpi=100, layout="constrained")
fig.suptitle(f'Half-hourly', fontsize=16)
heatmap(export_df['SW_IN_T1_47_1_gfXG'], ax=axs, title="SW_IN_T1_47_1_gfXG")

***

# 💾 Save to file

In [ ]:
filepath = dv.save_parquet(filename=OUTNAME, data=export_df, outpath=OUTPATH)
export_df.to_csv(Path(OUTPATH) / f"{OUTNAME}.csv")
print(f"Saved to: {filepath}")

***

# End of notebook.

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")